In [63]:
# Dataset: https://www.kaggle.com/datasets/rmisra/news-category-dataset

In [64]:
import pandas as pd
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense 
from sklearn.model_selection import train_test_split
import pickle

### Date gathering

In [65]:
data = pd.read_json('News_Category_Dataset_v3.json', lines=True)

### Data exploration

In [66]:
data

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22
...,...,...,...,...,...,...
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28


In [67]:
data.shape

(209527, 6)

In [68]:
data.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date'], dtype='str')

In [69]:
# We need only the short description for our next word prediction task. So we will drop the other columns.
data = data[['short_description']][:1000]

In [70]:
data

,short_description
0,Health experts said it is too early to predict...
1,He was subdued by passengers and crew when he ...
2,"""Until you have a dog you don't understand wha..."
3,"""Accidentally put grown-up toothpaste on my to..."
4,Amy Cooper accused investment firm Franklin Te...
...,...
995,The change does not appear imminent as the gov...
996,World leaders have been calling for an investi...
997,Assad's office said he met with Sheikh Mohamed...
998,The 7.4-magnitude temblor knocked out power an...


### Check null and duplicate columns and remove it

In [71]:
# Check null values
data.isnull().sum()

short_description    0
dtype: int64

In [72]:
# Check duplicates
data.duplicated().sum()

np.int64(0)

In [73]:
# Drop duplicates
data.drop_duplicates(inplace=True)

In [74]:
# Check duplicates after remove it
data.duplicated().sum()

np.int64(0)

### Date preprocessing

In [75]:
tokenizer = Tokenizer(oov_token='<OOV>')

In [76]:
# Fit the tokenizer on the short descriptions
tokenizer.fit_on_texts(data['short_description'])

In [77]:
tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'a': 3,
 'to': 4,
 'of': 5,
 'in': 6,
 'and': 7,
 'for': 8,
 'on': 9,
 'that': 10,
 'was': 11,
 'is': 12,
 'said': 13,
 'with': 14,
 'as': 15,
 'his': 16,
 'at': 17,
 'has': 18,
 'an': 19,
 'after': 20,
 'from': 21,
 'her': 22,
 'it': 23,
 'he': 24,
 'who': 25,
 'new': 26,
 'will': 27,
 'are': 28,
 'by': 29,
 'have': 30,
 'about': 31,
 '”': 32,
 'be': 33,
 'u': 34,
 'president': 35,
 's': 36,
 'former': 37,
 'more': 38,
 'this': 39,
 'one': 40,
 'people': 41,
 'two': 42,
 'their': 43,
 '—': 44,
 'you': 45,
 'year': 46,
 'were': 47,
 'not': 48,
 'trump': 49,
 'when': 50,
 'but': 51,
 'they': 52,
 'over': 53,
 'been': 54,
 'house': 55,
 'into': 56,
 'up': 57,
 'out': 58,
 'told': 59,
 'first': 60,
 'than': 61,
 'i': 62,
 'its': 63,
 'against': 64,
 'she': 65,
 'or': 66,
 'police': 67,
 'russian': 68,
 'say': 69,
 'how': 70,
 'star': 71,
 'also': 72,
 'last': 73,
 'what': 74,
 'could': 75,
 'had': 76,
 'biden': 77,
 'ukraine': 78,
 'white': 79,
 'court': 80,
 'sho

In [78]:
# Count of unique words in the dataset
# Add 1 because tokenizer indices start at 1, and 0 is reserved for padding
vocab_size=len(tokenizer.word_index) + 1

In [79]:
# Convert the short descriptions to sequences of integers
tokenizer.texts_to_sequences(data['short_description'])

[[131,
  249,
  13,
  23,
  12,
  286,
  355,
  4,
  2139,
  250,
  637,
  166,
  638,
  57,
  14,
  2,
  2140,
  101,
  851,
  5,
  2,
  26,
  2141,
  2,
  34,
  36,
  639,
  8,
  2,
  508],
 [24,
  11,
  2142,
  29,
  2143,
  7,
  2144,
  50,
  24,
  509,
  4,
  2,
  141,
  5,
  2,
  2145,
  20,
  2,
  2146,
  220,
  4,
  2,
  34,
  36,
  2147,
  142,
  6,
  640,
  641],
 [413, 45, 30, 3, 287, 45, 203, 2148, 74, 75, 33, 2149],
 [2150,
  510,
  2151,
  57,
  2152,
  9,
  114,
  2153,
  2154,
  7,
  24,
  2155,
  102,
  62,
  11,
  2156,
  16,
  1237,
  14,
  3,
  288,
  2157,
  2158,
  6,
  2159,
  2160],
 [1238,
  2161,
  121,
  2162,
  1239,
  2163,
  2164,
  5,
  2165,
  2166,
  22,
  7,
  2167,
  22,
  3,
  1240,
  20,
  251,
  5,
  2,
  1241,
  252,
  2168,
  221,
  511],
 [2,
  2169,
  46,
  103,
  253,
  11,
  512,
  642,
  17,
  2,
  643,
  288,
  1242,
  9,
  852,
  65,
  11,
  289,
  513,
  290,
  20,
  22,
  204,
  148,
  22,
  514,
  167,
  13],
 [1243,
  10,
  515,
  45,


In [80]:
input_sequences = []
for line in data['short_description']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [81]:
input_sequences

[[131, 249],
 [131, 249, 13],
 [131, 249, 13, 23],
 [131, 249, 13, 23, 12],
 [131, 249, 13, 23, 12, 286],
 [131, 249, 13, 23, 12, 286, 355],
 [131, 249, 13, 23, 12, 286, 355, 4],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250, 637],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250, 637, 166],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250, 637, 166, 638],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250, 637, 166, 638, 57],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250, 637, 166, 638, 57, 14],
 [131, 249, 13, 23, 12, 286, 355, 4, 2139, 250, 637, 166, 638, 57, 14, 2],
 [131,
  249,
  13,
  23,
  12,
  286,
  355,
  4,
  2139,
  250,
  637,
  166,
  638,
  57,
  14,
  2,
  2140],
 [131,
  249,
  13,
  23,
  12,
  286,
  355,
  4,
  2139,
  250,
  637,
  166,
  638,
  57,
  14,
  2,
  2140,
  101],
 [131,
  249,
  13,
  23,
  12,
  286,
  355,
  4,
  2139,
  250,
  637,
  166,
  638,
  57,
  14,
  2,

In [82]:
max_len = max(len(x) for x in input_sequences)

In [83]:
max_len

44

In [84]:
input = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [85]:
# All row 
X = input[:,:-1]
# Final word of each row
y = np.array(input[:,-1])

### Train Test split

In [86]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [87]:
X_train.shape, y_train.shape

((15442, 43), (15442,))

In [88]:
max_len = X.shape[1]

### Create a model

In [89]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    LSTM(256),
    Dense(vocab_size, activation='softmax')
])

/home/linux/Desktop/projects/demos/next-word-prediction-using-lstm/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [90]:
model.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
    )

In [91]:
model.build(input_shape=(None, max_len))

In [92]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 43, 128)        │       724,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5661)           │     1,454,877 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,573,725 (9.82 MB)

 Trainable params: 2,573,725 (9.82 MB)

 Non-trainable params: 0 (0.00 B)

In [93]:
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 70s 139ms/step - accuracy: 0.0547 - loss: 7.5819 - val_accuracy: 0.0601 - val_loss: 7.4644
Epoch 2/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 61s 126ms/step - accuracy: 0.0639 - loss: 6.9455 - val_accuracy: 0.0666 - val_loss: 7.5735
Epoch 3/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 70s 144ms/step - accuracy: 0.0757 - loss: 6.5949 - val_accuracy: 0.0717 - val_loss: 7.6952
Epoch 4/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 76s 131ms/step - accuracy: 0.0873 - loss: 6.2240 - val_accuracy: 0.0746 - val_loss: 7.8574
Epoch 5/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 66s 136ms/step - accuracy: 0.1005 - loss: 5.7358 - val_accuracy: 0.0730 - val_loss: 8.0954
Epoch 6/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 208s 431ms/step - accuracy: 0.1212 - loss: 5.1438 - val_accuracy: 0.0717 - val_loss: 8.3568
Epoch 7/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 65s 135ms/step - accuracy: 0.1574 - loss: 4.5079 - val_accuracy: 0.0697 - val_loss: 8.6422
Epoch 8/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 199s 157ms/step - accuracy: 0.2384 - loss:

In [94]:
model.save('nextword.h5')

In [95]:
with open('tokenzier.pkl', 'wb') as file:
    pickle.dump(tokenizer, file)